In [2]:
!pip install openai pandas numpy

In [3]:
import os
import json
import pandas as pd
import numpy as np
from openai import OpenAI

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

In [4]:
def build_funnel_dataset():
    np.random.seed(42)
    users = 10000

    traffic_source = np.random.choice(["ads", "organic"], users, p=[0.4, 0.6])
    device = np.random.choice(["mobile", "desktop"], users, p=[0.7, 0.3])
    experiment = np.random.choice(["A", "B"], users)
    date = pd.to_datetime("2025-01-01") + pd.to_timedelta(
        np.random.randint(0, 30, users), unit="D"
    )

    stage = []
    for _ in range(users):

        # Visit → Signup: 70% proceed, 30% stop at visit
        if np.random.rand() > 0.70:
            stage.append("visit")
            continue

        # Signup → Activation: 50% proceed, 50% stop at signup
        if np.random.rand() > 0.50:
            stage.append("signup")
            continue

        # Activation → Purchase: 40% proceed, 60% stop at activation
        if np.random.rand() > 0.40:
            stage.append("activation")
            continue

        stage.append("purchase")

    df = pd.DataFrame({
        "user_id": np.arange(users),
        "date": date,
        "traffic_source": traffic_source,
        "device": device,
        "experiment": experiment,
        "stage_reached": stage
    })

    # Inject hidden anomaly — mobile checkout broke after Jan 20
    mask = (
        (df["device"] == "mobile") &
        (df["stage_reached"] == "purchase") &
        (df["date"] > "2025-01-20")
    )
    df.loc[mask, "stage_reached"] = "activation"

    stage_order = {"visit": 1, "signup": 2, "activation": 3, "purchase": 4}
    df["stage_number"] = df["stage_reached"].map(stage_order)

    return df

df = build_funnel_dataset()
print("Dataset built:", df.shape)
print(df["stage_reached"].value_counts())

# This shows the CUMULATIVE funnel — how many reached AT LEAST each stage
print("\n--- Cumulative Funnel (how a PM reads it) ---")
print("Visits:     ", (df["stage_number"] >= 1).sum())
print("Signups:    ", (df["stage_number"] >= 2).sum())
print("Activations:", (df["stage_number"] >= 3).sum())
print("Purchases:  ", (df["stage_number"] >= 4).sum())

Dataset built: (10000, 7)
stage_reached
signup        3465
visit         2986
activation    2470
purchase      1079
Name: count, dtype: int64

--- Cumulative Funnel (how a PM reads it) ---
Visits:      10000
Signups:     7014
Activations: 3549
Purchases:   1079


In [5]:
# ============================================================
# CELL 4 — Tool Functions
# ------------------------------------------------------------
# These are regular Python functions.
# The LLM cannot run these directly.
# It will REQUEST them, and our agent loop (Cell 7) will
# execute them and return results back to the LLM.
# ============================================================


# ------------------------------------------------------------
# TOOL 1 — Overall Funnel Summary
# ------------------------------------------------------------
# The agent should call this FIRST
# It gives the big picture before drilling into specifics
# ------------------------------------------------------------

def get_funnel_summary() -> str:

    total = len(df)

    # Count how many users reached AT LEAST each stage
    # This is the cumulative view — same as what we printed in Cell 3
    visits      = total
    signups     = int((df["stage_number"] >= 2).sum())
    activations = int((df["stage_number"] >= 3).sum())
    purchases   = int((df["stage_number"] >= 4).sum())

    # Calculate conversion rate at each step
    # Round to 3 decimal places for clean output
    summary = {
        "total_users"                   : visits,
        "signups"                       : signups,
        "activations"                   : activations,
        "purchases"                     : purchases,
        "visit_to_signup_rate"          : round(signups / visits, 3),
        "signup_to_activation_rate"     : round(activations / signups, 3),
        "activation_to_purchase_rate"   : round(purchases / activations, 3),
        "overall_conversion_rate"       : round(purchases / visits, 3)
    }

    # We return JSON string, not a Python dict
    # Because the LLM receives text, not Python objects
    return json.dumps(summary)


# ------------------------------------------------------------
# TOOL 2 — Anomaly Detection
# ------------------------------------------------------------
# Calculates daily activation→purchase rate
# Flags days where rate drops 30% below the average
# Agent calls this to find WHEN the problem started
# ------------------------------------------------------------

def detect_anomalies() -> str:

    # Group data by date — one row per day
    daily = df.groupby("date").agg(
        activations=("stage_number", lambda x: (x >= 3).sum()),
        purchases  =("stage_number", lambda x: (x >= 4).sum())
    ).reset_index()

    # Calculate daily activation→purchase conversion rate
    # replace(0, np.nan) prevents division by zero on days with no activations
    daily["rate"] = daily["purchases"] / daily["activations"].replace(0, np.nan)

    # Baseline: what is the average rate across all days?
    avg_rate = daily["rate"].mean()

    # Threshold: flag any day that drops more than 30% below average
    threshold = avg_rate * 0.70

    # Filter to only anomaly days
    anomalies = daily[daily["rate"] < threshold].copy()

    # Convert dates to string so they can be JSON serialised
    anomalies["date"] = anomalies["date"].astype(str)

    result = {
        "average_activation_purchase_rate" : round(avg_rate, 3),
        "anomaly_threshold"                : round(threshold, 3),
        "anomaly_days_count"               : len(anomalies),
        "anomaly_dates"                    : anomalies["date"].tolist(),
        "anomaly_rates"                    : anomalies["rate"].round(3).tolist()
    }

    return json.dumps(result)


# ------------------------------------------------------------
# TOOL 3 — Device Breakdown
# ------------------------------------------------------------
# Splits conversion rate by mobile vs desktop
# Accepts optional date filters so agent can focus on
# the anomaly period it discovered in Tool 2
# ------------------------------------------------------------

def get_device_breakdown(start_date: str = None, end_date: str = None) -> str:

    filtered = df.copy()

    # Apply date filters if the agent passes them
    if start_date:
        filtered = filtered[filtered["date"] >= start_date]
    if end_date:
        filtered = filtered[filtered["date"] <= end_date]

    breakdown = {}

    for device in ["mobile", "desktop"]:

        # Filter to just this device type
        d = filtered[filtered["device"] == device]

        activations = int((d["stage_number"] >= 3).sum())
        purchases   = int((d["stage_number"] >= 4).sum())

        # Avoid division by zero
        rate = round(purchases / activations, 3) if activations > 0 else 0

        breakdown[device] = {
            "activations"              : activations,
            "purchases"                : purchases,
            "activation_purchase_rate" : rate
        }

    return json.dumps(breakdown)


# ------------------------------------------------------------
# TOOL 4 — Traffic Source Breakdown
# ------------------------------------------------------------
# Splits conversion rate by ads vs organic
# Helps agent rule out acquisition-related causes
# Same date filter pattern as Tool 3
# ------------------------------------------------------------

def get_traffic_source_breakdown(start_date: str = None, end_date: str = None) -> str:

    filtered = df.copy()

    if start_date:
        filtered = filtered[filtered["date"] >= start_date]
    if end_date:
        filtered = filtered[filtered["date"] <= end_date]

    breakdown = {}

    for source in ["ads", "organic"]:

        d = filtered[filtered["traffic_source"] == source]

        activations = int((d["stage_number"] >= 3).sum())
        purchases   = int((d["stage_number"] >= 4).sum())

        rate = round(purchases / activations, 3) if activations > 0 else 0

        breakdown[source] = {
            "activations"              : activations,
            "purchases"                : purchases,
            "activation_purchase_rate" : rate
        }

    return json.dumps(breakdown)


# ------------------------------------------------------------
# Quick test — run each tool manually to verify they work
# before connecting them to the agent
# ------------------------------------------------------------

print("=== TOOL 1: Funnel Summary ===")
print(get_funnel_summary())

print("\n=== TOOL 2: Anomaly Detection ===")
print(detect_anomalies())

print("\n=== TOOL 3: Device Breakdown (full period) ===")
print(get_device_breakdown())

print("\n=== TOOL 3: Device Breakdown (after Jan 20 only) ===")
print(get_device_breakdown(start_date="2025-01-20"))

print("\n=== TOOL 4: Traffic Source Breakdown ===")
print(get_traffic_source_breakdown())

=== TOOL 1: Funnel Summary ===
{"total_users": 10000, "signups": 7014, "activations": 3549, "purchases": 1079, "visit_to_signup_rate": 0.701, "signup_to_activation_rate": 0.506, "activation_to_purchase_rate": 0.304, "overall_conversion_rate": 0.108}

=== TOOL 2: Anomaly Detection ===
{"average_activation_purchase_rate": 0.306, "anomaly_threshold": 0.214, "anomaly_days_count": 10, "anomaly_dates": ["2025-01-21", "2025-01-22", "2025-01-23", "2025-01-24", "2025-01-25", "2025-01-26", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30"], "anomaly_rates": [0.161, 0.079, 0.13, 0.158, 0.125, 0.129, 0.051, 0.159, 0.156, 0.105]}

=== TOOL 3: Device Breakdown (full period) ===
{"mobile": {"activations": 2454, "purchases": 635, "activation_purchase_rate": 0.259}, "desktop": {"activations": 1095, "purchases": 444, "activation_purchase_rate": 0.405}}

=== TOOL 3: Device Breakdown (after Jan 20 only) ===
{"mobile": {"activations": 930, "purchases": 39, "activation_purchase_rate": 0.042}, "desktop"

In [6]:
# ============================================================
# CELL 5 — Tool Registry (JSON Schemas)
# ------------------------------------------------------------
# KEY CONCEPT:
# The LLM lives on OpenAI's servers. It cannot see your Python
# functions. The only way it knows your tools exist is through
# these JSON descriptions that you send with every API call.
#
# The LLM reads:
#   - "name" → what to call when requesting this tool
#   - "description" → WHEN to use this tool (most important)
#   - "parameters" → what arguments to pass
#
# If your description is vague, the LLM picks the wrong tool.
# If your description is clear, the LLM picks the right tool.
# Description quality is a PM-level skill — not a coding skill.
# ============================================================

TOOLS = [

    # --------------------------------------------------------
    # TOOL 1 — Funnel Summary
    # --------------------------------------------------------
    {
        "type": "function",
        "function": {
            "name": "get_funnel_summary",

            # This description tells the LLM WHEN to call this tool
            # Notice: "Call this FIRST" — we're guiding the agent's
            # sequencing through the description itself
            "description": (
                "Returns overall funnel conversion rates at each stage: "
                "visit to signup, signup to activation, activation to purchase. "
                "Also returns total user counts at each stage. "
                "Call this FIRST to understand baseline funnel performance "
                "before investigating specific problems."
            ),

            # This tool needs no parameters — it always returns
            # the full dataset summary
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },

    # --------------------------------------------------------
    # TOOL 2 — Anomaly Detection
    # --------------------------------------------------------
    {
        "type": "function",
        "function": {
            "name": "detect_anomalies",

            "description": (
                "Detects specific dates where the activation to purchase "
                "conversion rate dropped abnormally — more than 30 percent "
                "below the average rate. Returns the anomaly dates, their "
                "conversion rates, and the normal baseline rate. "
                "Call this to find WHEN a conversion problem started."
            ),

            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },

    # --------------------------------------------------------
    # TOOL 3 — Device Breakdown
    # --------------------------------------------------------
    {
        "type": "function",
        "function": {
            "name": "get_device_breakdown",

            "description": (
                "Returns activation to purchase conversion rates split by "
                "device type: mobile vs desktop. "
                "Accepts optional date filters to focus on a specific time period. "
                "Call this to determine whether a conversion problem affects "
                "mobile users, desktop users, or both equally. "
                "Use start_date from anomaly detection results to filter "
                "to the affected period."
            ),

            # This tool accepts optional parameters
            # The LLM will fill these in based on what it has learned
            # from previous tool calls — like anomaly dates from Tool 2
            "parameters": {
                "type": "object",
                "properties": {
                    "start_date": {
                        "type": "string",
                        "description": (
                            "Filter data from this date onwards. "
                            "Format: YYYY-MM-DD. Optional."
                        )
                    },
                    "end_date": {
                        "type": "string",
                        "description": (
                            "Filter data up to this date. "
                            "Format: YYYY-MM-DD. Optional."
                        )
                    }
                },
                "required": []
            }
        }
    },

    # --------------------------------------------------------
    # TOOL 4 — Traffic Source Breakdown
    # --------------------------------------------------------
    {
        "type": "function",
        "function": {
            "name": "get_traffic_source_breakdown",

            "description": (
                "Returns activation to purchase conversion rates split by "
                "traffic source: ads vs organic. "
                "Accepts optional date filters. "
                "Call this to check whether a conversion problem is caused "
                "by a specific acquisition channel or affects all traffic equally. "
                "If both channels are equally affected, the problem is likely "
                "a product or technical issue, not a marketing issue."
            ),

            "parameters": {
                "type": "object",
                "properties": {
                    "start_date": {
                        "type": "string",
                        "description": (
                            "Filter data from this date onwards. "
                            "Format: YYYY-MM-DD. Optional."
                        )
                    },
                    "end_date": {
                        "type": "string",
                        "description": (
                            "Filter data up to this date. "
                            "Format: YYYY-MM-DD. Optional."
                        )
                    }
                },
                "required": []
            }
        }
    }
]

# --------------------------------------------------------
# Verify the registry loaded correctly
# --------------------------------------------------------
print(f"✅ Tool registry loaded: {len(TOOLS)} tools registered")
for tool in TOOLS:
    print(f"   - {tool['function']['name']}: {tool['function']['description'][:60]}...")

✅ Tool registry loaded: 4 tools registered
   - get_funnel_summary: Returns overall funnel conversion rates at each stage: visit...
   - detect_anomalies: Detects specific dates where the activation to purchase conv...
   - get_device_breakdown: Returns activation to purchase conversion rates split by dev...
   - get_traffic_source_breakdown: Returns activation to purchase conversion rates split by tra...


In [7]:
# ============================================================
# CELL 6 — Tool Dispatcher
# ------------------------------------------------------------
# KEY CONCEPT:
# The LLM cannot run Python. When it decides to use a tool,
# it returns a structured request — just text describing
# which function to call and with what arguments.
#
# This dispatcher:
#   1. Receives the LLM's tool request
#   2. Figures out which Python function to run
#   3. Executes it with the correct arguments
#   4. Returns the result as a string back to the agent loop
#
# This is the bridge between LLM decisions and real execution.
# ============================================================

def dispatch_tool(tool_name: str, tool_args: dict) -> str:
    """
    Executes the requested tool and returns its output.

    Args:
        tool_name: Name of the tool the LLM requested
        tool_args: Arguments the LLM wants to pass to the tool

    Returns:
        Tool output as a JSON string
    """

    # Log what's happening so we can follow the agent's actions
    # This is the beginning of our reasoning trace
    print(f"\n  ⚙️  TOOL CALLED  : {tool_name}")
    print(f"  📥 ARGUMENTS    : {tool_args}")

    # --------------------------------------------------------
    # Match the tool name to the actual Python function
    # and execute it with the arguments the LLM provided
    # --------------------------------------------------------

    if tool_name == "get_funnel_summary":
        # No arguments needed for this tool
        result = get_funnel_summary()

    elif tool_name == "detect_anomalies":
        # No arguments needed for this tool
        result = detect_anomalies()

    elif tool_name == "get_device_breakdown":
        # LLM may or may not pass date filters
        # **tool_args unpacks the dict as keyword arguments
        # e.g. {"start_date": "2025-01-20"} becomes start_date="2025-01-20"
        result = get_device_breakdown(**tool_args)

    elif tool_name == "get_traffic_source_breakdown":
        result = get_traffic_source_breakdown(**tool_args)

    else:
        # Safety net — if LLM hallucinates a tool that doesn't exist
        # We return an error message instead of crashing
        result = json.dumps({
            "error": f"Tool '{tool_name}' does not exist.",
            "available_tools": [
                "get_funnel_summary",
                "detect_anomalies",
                "get_device_breakdown",
                "get_traffic_source_breakdown"
            ]
        })

    # Show a preview of what the tool returned
    print(f"  📤 RESULT PREVIEW: {result[:120]}...")

    return result


# --------------------------------------------------------
# Quick test — simulate what happens when LLM requests a tool
# --------------------------------------------------------

print("=== DISPATCHER TEST 1: Valid tool, no arguments ===")
print(dispatch_tool("get_funnel_summary", {}))

print("\n=== DISPATCHER TEST 2: Valid tool, with arguments ===")
print(dispatch_tool("get_device_breakdown", {"start_date": "2025-01-20"}))

print("\n=== DISPATCHER TEST 3: Hallucinated tool (safety net) ===")
print(dispatch_tool("get_experiment_breakdown", {}))

=== DISPATCHER TEST 1: Valid tool, no arguments ===

  ⚙️  TOOL CALLED  : get_funnel_summary
  📥 ARGUMENTS    : {}
  📤 RESULT PREVIEW: {"total_users": 10000, "signups": 7014, "activations": 3549, "purchases": 1079, "visit_to_signup_rate": 0.701, "signup_t...
{"total_users": 10000, "signups": 7014, "activations": 3549, "purchases": 1079, "visit_to_signup_rate": 0.701, "signup_to_activation_rate": 0.506, "activation_to_purchase_rate": 0.304, "overall_conversion_rate": 0.108}

=== DISPATCHER TEST 2: Valid tool, with arguments ===

  ⚙️  TOOL CALLED  : get_device_breakdown
  📥 ARGUMENTS    : {'start_date': '2025-01-20'}
  📤 RESULT PREVIEW: {"mobile": {"activations": 930, "purchases": 39, "activation_purchase_rate": 0.042}, "desktop": {"activations": 430, "pu...
{"mobile": {"activations": 930, "purchases": 39, "activation_purchase_rate": 0.042}, "desktop": {"activations": 430, "purchases": 171, "activation_purchase_rate": 0.398}}

=== DISPATCHER TEST 3: Hallucinated tool (safety net) ===

 

In [8]:
# ============================================================
# CELL 7 — The ReAct Agent Loop
# ------------------------------------------------------------
# KEY CONCEPT:
# This is what makes the system genuinely agentic.
# The LLM drives the sequence — not your Python code.
# Your Python code just:
#   1. Sends messages to the LLM
#   2. Executes whatever tool the LLM requests
#   3. Adds results back to conversation history
#   4. Repeats until LLM gives a final answer
# ============================================================

def run_agent(user_goal: str, max_iterations: int = 10) -> str:
    """
    Runs the ReAct agent loop.

    Args:
        user_goal     : The investigation goal in plain English
        max_iterations: Safety limit to prevent infinite loops

    Returns:
        The agent's final answer as a string
    """

    print("\n" + "="*60)
    print("🤖 AGENT STARTED")
    print(f"🎯 GOAL: {user_goal}")
    print("="*60)

    # --------------------------------------------------------
    # The conversation history
    # This list grows with every iteration
    # The LLM sees the entire history on every API call
    # --------------------------------------------------------
    messages = [

        # System prompt — defines the agent's role and behaviour
        # This is the agent's "personality" and "instructions"
        {
            "role": "system",
            "content": (
                "You are a senior product analytics agent. "
                "Your job is to investigate funnel performance problems systematically. "
                "Always start by getting the funnel summary to understand baseline. "
                "Then detect anomalies to find when the problem started. "
                "Then drill into segments like device and traffic source to find who is affected. "
                "Use date filters from anomaly results when calling segmentation tools. "
                "Be methodical. Use all relevant tools before concluding. "
                "Your final answer should clearly state: what the problem is, "
                "when it started, which segment is affected, and the most likely root cause."
            )
        },

        # The user's goal — plain English investigation request
        {
            "role": "user",
            "content": user_goal
        }
    ]

    # --------------------------------------------------------
    # Reasoning trace — collect every decision the agent makes
    # This is what you'll review to understand agent behaviour
    # --------------------------------------------------------
    reasoning_trace = []
    iteration = 0

    # ============================================================
    # THE AGENT LOOP
    # ============================================================
    while iteration < max_iterations:
        iteration += 1

        print(f"\n{'─'*40}")
        print(f"🔄 ITERATION {iteration}")
        print(f"{'─'*40}")
        print(f"📨 Sending {len(messages)} messages to LLM...")

        # --------------------------------------------------------
        # Send full conversation history to LLM
        # The LLM sees everything that happened before
        # --------------------------------------------------------
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,          # The tool schemas from Cell 5
            tool_choice="auto",   # LLM decides: call tool OR answer
            temperature=0         # Deterministic output for reliability
        )

        message      = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        print(f"🧠 LLM DECISION: {finish_reason}")

        # ========================================================
        # CASE 1: LLM wants to call a tool
        # finish_reason == "tool_calls" means LLM is not done yet
        # It needs more information from a tool
        # ========================================================
        if finish_reason == "tool_calls":

            # Add LLM's decision to conversation history
            # Important: this must happen BEFORE adding tool results
            messages.append(message)

            # LLM may request multiple tools in one iteration
            for tool_call in message.tool_calls:

                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                print(f"🔧 TOOL REQUESTED: {tool_name}")
                print(f"📋 WITH ARGS     : {tool_args}")

                # Log this reasoning step
                reasoning_trace.append({
                    "iteration" : iteration,
                    "action"    : "tool_call",
                    "tool"      : tool_name,
                    "args"      : tool_args
                })

                # ------------------------------------------------
                # Execute the tool via dispatcher
                # This is where Python actually runs the function
                # ------------------------------------------------
                tool_result = dispatch_tool(tool_name, tool_args)

                # Log the observation
                reasoning_trace.append({
                    "iteration"      : iteration,
                    "action"         : "observation",
                    "tool"           : tool_name,
                    "result_preview" : tool_result[:80]
                })

                # ------------------------------------------------
                # Add tool result back to conversation history
                # This is how the LLM "sees" what the tool returned
                # Without this step the LLM would never know the result
                # ------------------------------------------------
                messages.append({
                    "role"         : "tool",
                    "tool_call_id" : tool_call.id,
                    "content"      : tool_result
                })

            # Loop continues — LLM will see the new tool results
            # on the next iteration and decide what to do next

        # ========================================================
        # CASE 2: LLM is done — gives final answer
        # finish_reason == "stop" means LLM has enough information
        # ========================================================
        elif finish_reason == "stop":

            final_answer = message.content

            print(f"\n{'='*60}")
            print("✅ AGENT FINISHED")
            print(f"{'='*60}")
            print(final_answer)

            # Log the final step
            reasoning_trace.append({
                "iteration" : iteration,
                "action"    : "final_answer",
                "preview"   : final_answer[:100]
            })

            # ------------------------------------------------
            # Print the full reasoning trace
            # This shows you exactly how the agent thought
            # ------------------------------------------------
            print(f"\n{'='*60}")
            print("📋 REASONING TRACE (how the agent thought):")
            print(f"{'='*60}")
            for i, step in enumerate(reasoning_trace):
                print(f"\nStep {i+1}: {step['action'].upper()}")
                if step['action'] == 'tool_call':
                    print(f"  Tool : {step['tool']}")
                    print(f"  Args : {step['args']}")
                elif step['action'] == 'observation':
                    print(f"  Tool   : {step['tool']}")
                    print(f"  Result : {step['result_preview']}")
                elif step['action'] == 'final_answer':
                    print(f"  Answer : {step['preview']}")

            return final_answer

        else:
            print(f"⚠️ Unexpected finish reason: {finish_reason}")
            break

    return "Agent reached maximum iterations without completing."


# ============================================================
# CELL 7 ONLY DEFINES THE FUNCTION — does not run the agent yet
# We run the agent in Cell 8
# ============================================================
print("✅ Agent loop defined successfully")
print("Run Cell 8 to start the agent")

✅ Agent loop defined successfully
Run Cell 8 to start the agent


In [9]:
# ============================================================
# CELL 8 — Run the Agent
# ------------------------------------------------------------
# This is all you write to start the investigation.
# The agent decides everything else.
# ============================================================

result = run_agent(
    "Our purchase conversion has dropped significantly. "
    "Investigate the funnel, identify when the drop started, "
    "which user segments are affected, "
    "and what the most likely root cause is."
)


🤖 AGENT STARTED
🎯 GOAL: Our purchase conversion has dropped significantly. Investigate the funnel, identify when the drop started, which user segments are affected, and what the most likely root cause is.

────────────────────────────────────────
🔄 ITERATION 1
────────────────────────────────────────
📨 Sending 2 messages to LLM...
🧠 LLM DECISION: tool_calls
🔧 TOOL REQUESTED: get_funnel_summary
📋 WITH ARGS     : {}

  ⚙️  TOOL CALLED  : get_funnel_summary
  📥 ARGUMENTS    : {}
  📤 RESULT PREVIEW: {"total_users": 10000, "signups": 7014, "activations": 3549, "purchases": 1079, "visit_to_signup_rate": 0.701, "signup_t...

────────────────────────────────────────
🔄 ITERATION 2
────────────────────────────────────────
📨 Sending 4 messages to LLM...
🧠 LLM DECISION: tool_calls
🔧 TOOL REQUESTED: detect_anomalies
📋 WITH ARGS     : {}

  ⚙️  TOOL CALLED  : detect_anomalies
  📥 ARGUMENTS    : {}
  📤 RESULT PREVIEW: {"average_activation_purchase_rate": 0.306, "anomaly_threshold": 0.214, "anomaly_d